In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import torch

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.local.localization.helpers import (
    clone_bussam,
    download_sam_checkpoint,
    prepare_bussam_busi_dataset,
    patch_bussam_config,
    train_bussam,
    patch_bussam_test_config,
    find_latest_checkpoint,
    test_bussam,
)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device -', device)

if device == 'cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
'''
I referred to the BUSSAM repo.
See: https://github.com/bscs12/BUSSAM

After classification we train a SAM-based model called BUSSAM for lesion segmentation/localisation.
The project two stages (classification and localisation) - where we test VLM variants and then BUSSAM training/testing for localisation.
This quickly allows medical professionals tofind the lesion region after the image has been classified as either benign or malignant.
'''

clone_bussam()

In [ ]:
download_sam_checkpoint()

In [ ]:
dataset_info = prepare_bussam_busi_dataset()
dataset_info

In [ ]:
patch_bussam_config(epochs=100, output_dir="outputs/")

In [ ]:
train_results = train_bussam(batch_size=4, base_lr=0.0005)
history = train_results['history']
best_checkpoint = train_results['best_checkpoint'] or find_latest_checkpoint()
patch_bussam_test_config(best_checkpoint)

epochs = [row['epoch'] for row in history]
train_loss = [row['train_loss'] for row in history]
val_loss = [row['val_loss'] for row in history]
val_dice = [row['val_dice'] for row in history]
val_accuracy = [row['val_accuracy'] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_loss, label='Train loss')
axes[0].plot(epochs, val_loss, label='Val loss')
axes[0].set_title('BUSSAM loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_dice, label='Val dice')
axes[1].plot(epochs, val_accuracy, label='Val accuracy')
axes[1].set_title('BUSSAM validation metrics')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Percent')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

best_checkpoint

In [ ]:
test_bussam(batch_size=4)